In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!mkdir -p data/tok32k_wnmix
!curl -L --retry 3 -o data/tok32k_wnmix/meta.json https://github.com/shemshallah/nanlmv9/releases/download/wt103-bytefirst/wnmix_meta.json
!for p in aa ab ac ad; do curl -L --retry 3 -o cmwt.part-$p https://github.com/shemshallah/nanlmv9/releases/download/wt103-bytefirst/cmwt.part-$p; done
!cat cmwt.part-* > cm_wt103_prefix_512_ctx17.bin
!curl -L --retry 3 -o win_wt103_8m.bin https://github.com/shemshallah/nanlmv9/releases/download/wt103-bytefirst/win_wt103_8m.bin
!for p in aa ab ac ad ae af; do curl -L --retry 3 -o cm17.part-$p https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/cm17.part-$p; done
!cat cm17.part-* > cm_valid_full_ctx17.bin
!curl -L --retry 3 -o win_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-ctx17/win_valid_full.bin
!ls -lh cm_wt103_prefix_512_ctx17.bin win_wt103_8m.bin cm_valid_full_ctx17.bin

In [ ]:
!python3 experiments/mamba_moe/train_word_char.py \
  --data-dir data/tok32k_wnmix --cm-dump cm_wt103_prefix_512_ctx17.bin --window win_wt103_8m.bin \
  --cm-drop 0.1 --gate-entropy 0.01 --gate-max-cm 0.9 \
  --eval-window win_valid_full.bin --eval-cm-dump cm_valid_full_ctx17.bin --eval-chunks 20 \
  --out runs/d256_wt103 --steps 100000 --tokens 64 \
  --d-model 256 --char-model 64 --experts 64 --lr 2e-4 \
  --scan chunked --scan-chunk 4 --device cuda --amp --max-seconds 600 \
  --save-interval 500 --result-json runs/d256_wt103/result.json

In [ ]:
import json; print(open('runs/d256_wt103/result.json').read())